# Crop Yield Forecasting — Experimentation Notebook

**Purpose.** Iterate on crop yield models (corn, soybean, wheat) against the same data + baselines the production backend uses, log every experiment to MLflow, and emit artifacts that can be dropped directly into `backend/artifacts/yield/{crop}/week_{N}/`.

**Contract with the production backend.** If your model:
1. Pickles as a `backend.models.yield_model.YieldModel` instance (use the helper `save_backend_compatible_artifact()` below — it handles this for you),
2. Beats the county-historical-mean RRMSE by ≥ 10% on the held-out test set,
3. Emits the canonical `metrics.json` alongside,

…it is adoptable. The maintainer will sign it, upload to S3, and restart the FastAPI service.

**What this notebook does (and does not) do.**
- ✅ Reads raw NASS + GHCN weather + PRISM normals from S3.
- ✅ Builds the canonical 10-feature vector (5 NASS historical + 5 weather).
- ✅ Offers a model factory with swappable backends (LightGBM, XGBoost, CatBoost, RandomForest, HistGradientBoosting, or your own).
- ✅ Walk-forward splits (train ≤2019, val 2020–2022, test 2023–2024) matching production.
- ✅ Logs params, metrics, top-feature importances, and the pickled model to MLflow.
- ✅ Enforces the baseline gate before exporting.
- ❌ Does not touch the production RDS. DB writes happen in the adoption step.
- ❌ Does not require `MODEL_SIGNING_KEY`. Signing happens in adoption.

---

## 1 · Setup

Run the next three cells once. Total setup time: ~90s in Colab.

In [ ]:
# 1.1 — Install dependencies
# Colab already has pandas/numpy/scikit-learn; we add the model backends and MLflow.
%pip install -q \
    'lightgbm>=4.0' \
    'xgboost>=2.0' \
    'catboost>=1.2' \
    'mlflow>=2.10' \
    'boto3' \
    'pyarrow' \
    'fastparquet' \
    'scikit-learn>=1.4' 2>&1 | tail -n 3

In [ ]:
# 1.2 — Clone the repo (so we can import the YieldModel class that production expects)
# Change REPO_URL to your fork if collaborating via PR.
import os, sys, subprocess

REPO_URL = os.environ.get('ADS_REPO_URL', 'https://github.com/YOUR_ORG/Agricultural_Data_Analysis.git')
REPO_DIR = '/content/Agricultural_Data_Analysis' if os.path.isdir('/content') else './Agricultural_Data_Analysis'

if not os.path.isdir(REPO_DIR):
    # Fallback: if we're already inside the repo (local dev), just use cwd
    if os.path.isfile('backend/models/yield_model.py'):
        REPO_DIR = os.getcwd()
    else:
        print(f'Cloning {REPO_URL} -> {REPO_DIR}')
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('Repo path:', REPO_DIR)
print('YieldModel importable:', os.path.isfile(f'{REPO_DIR}/backend/models/yield_model.py'))

In [ ]:
# 1.3 — Imports
import json, pickle, time, hashlib, tempfile, warnings
from dataclasses import dataclass, field
from datetime import date, timedelta
from pathlib import Path
from typing import Any, Callable

import numpy as np
import pandas as pd
import mlflow

warnings.filterwarnings('ignore', category=UserWarning)
pd.set_option('display.max_columns', 60)

# Import the production YieldModel — this is the class our exported pickle must be an instance of.
# If the import fails, check cell 1.2 above.
from backend.models.yield_model import YieldModel, compute_baselines, compute_rrmse, FEATURE_LABELS
print('Imported YieldModel. Feature labels recognized by backend:', list(FEATURE_LABELS.keys())[:5], '...')

## 2 · Configuration

**This is the cell you edit** to pick a crop, weeks of interest, model family, and hyperparameters. Every subsequent cell reads from `CONFIG` — so re-running from here down is all you need to re-run a new experiment.

In [ ]:
# ========== USER-EDITABLE CONFIG ==========
CONFIG = {
    # Which crop to model: 'corn' | 'soybean' | 'wheat'
    'crop': 'corn',

    # Which weeks of the growing season to train (1..20).
    # Week 1 = first week after planting; week 20 ≈ late harvest. Production trains all 20.
    # For fast iteration, start with one representative week (10 = mid-season).
    'weeks': [10],

    # Model family: see MODEL_FACTORY in section 6. Add your own there.
    'model_family': 'lightgbm_quantile',

    # Hyperparameters — passed verbatim to the model builder in MODEL_FACTORY.
    # Change these freely; they are logged to MLflow so you can A/B later.
    'hyperparams': {
        'n_estimators': 500,
        'max_depth': 5,
        'learning_rate': 0.04,
        'min_child_samples': 20,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'reg_alpha': 0.1,
    },

    # Whether to include weather features. Set False to train a pure-NASS baseline.
    'use_weather_features': True,

    # Training splits — keep aligned with production unless you have a reason.
    'train_end_year': 2019,
    'val_years': (2020, 2022),
    'test_years': (2023, 2024),

    # Baseline gate: model must beat county_mean_rrmse by this pct to be adoptable.
    'baseline_gate_pct': 10.0,

    # Data source — these default to the public-read USDA S3 bucket.
    'data': {
        's3_bucket': 'usda-analysis-datasets',
        'nass_prefix': 'survey_datasets/partitioned_states_counties/',
        'weather_key': 'weather/ghcn_processed/county_weather_2000_2025.parquet',
        'prism_key': 'weather/prism_normals.csv',
        'aws_region': 'us-east-2',
    },

    # MLflow experiment name — defaults to 'yield_{crop}'.
    'mlflow_experiment': None,

    # Tag who's running the experiment — surfaces in the metrics.json experiment_meta block.
    'experimenter': os.environ.get('USER', 'anonymous'),
}
# ==========================================

# Derived constants
CROP = CONFIG['crop']
CROP_NASS = {'corn': 'CORN', 'soybean': 'SOYBEANS', 'wheat': 'WHEAT'}[CROP]
PLANTING_DOY = {'corn': 110, 'soybean': 125, 'wheat': 60}[CROP]
CONFIG['mlflow_experiment'] = CONFIG['mlflow_experiment'] or f'yield_{CROP}'

# Pretty-print the effective config for the run log
print(json.dumps({k: v for k, v in CONFIG.items() if k != 'data'}, indent=2, default=str))

In [ ]:
# 2.1 — MLflow init
#
# Precedence: if MLFLOW_TRACKING_URI is set in the environment we use that (e.g. DagsHub,
# Databricks, a self-hosted mlflow server). Otherwise we log locally to ./mlruns.
#
# To use DagsHub: set the three env vars below in a previous cell BEFORE this one runs.
#     os.environ['MLFLOW_TRACKING_URI']      = 'https://dagshub.com/<user>/<repo>.mlflow'
#     os.environ['MLFLOW_TRACKING_USERNAME'] = '<user>'
#     os.environ['MLFLOW_TRACKING_PASSWORD'] = '<dagshub token>'

tracking_uri = os.environ.get('MLFLOW_TRACKING_URI')
if tracking_uri:
    print(f'[MLflow] Remote tracking: {tracking_uri}')
    mlflow.set_tracking_uri(tracking_uri)
else:
    local = Path('./mlruns').absolute()
    print(f'[MLflow] Local tracking: {local}')
    mlflow.set_tracking_uri(f'file://{local}')

mlflow.set_experiment(CONFIG['mlflow_experiment'])
print(f'[MLflow] Experiment: {CONFIG["mlflow_experiment"]}')

# Turn on autologging for whichever backend we're using. Works for sklearn, lgbm, xgboost.
# Note: we still call mlflow.log_* manually to guarantee a consistent schema.
mlflow.sklearn.autolog(disable=True)  # don't pollute with extra child runs

## 3 · Data loading

Three data sources, all from S3:

| Source | What it is | Rows |
|---|---|---|
| NASS county parquets | County-level yield history 2000–2025 | ~870K |
| GHCN processed | Daily TMAX/TMIN/PRCP aggregated to county | ~16M |
| PRISM normals | 30-yr monthly precipitation normals by county | ~40K |

In [ ]:
# 3.1 — Lightweight S3 client. Uses unsigned requests for the public bucket unless AWS creds are set.
import boto3
from botocore import UNSIGNED
from botocore.client import Config

def make_s3_client():
    region = CONFIG['data']['aws_region']
    # If the user set AWS_ACCESS_KEY_ID, use it — otherwise go unsigned (public-read).
    if os.environ.get('AWS_ACCESS_KEY_ID'):
        return boto3.client('s3', region_name=region)
    return boto3.client('s3', region_name=region, config=Config(signature_version=UNSIGNED))

s3 = make_s3_client()
print('S3 client ready (signed)' if os.environ.get('AWS_ACCESS_KEY_ID') else 'S3 client ready (unsigned / public-read)')

In [ ]:
# 3.2 — Load NASS county yields
#
# Strategy: sync the partitioned parquet prefix to a local /tmp cache, then read.
# ~3-5 minutes for a full sync in Colab; cached between cells.

CACHE = Path(tempfile.gettempdir()) / 'ads_yield_cache'
CACHE.mkdir(exist_ok=True)

def _sync_prefix(bucket: str, prefix: str, dest: Path) -> int:
    """Download every object under prefix to dest. Returns count."""
    dest.mkdir(parents=True, exist_ok=True)
    paginator = s3.get_paginator('list_objects_v2')
    n = 0
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get('Contents', []):
            key = obj['Key']
            if key.endswith('/'):
                continue
            local = dest / Path(key).name
            if local.exists() and local.stat().st_size == obj['Size']:
                n += 1
                continue
            s3.download_file(bucket, key, str(local))
            n += 1
    return n

def load_nass_county_yields(crop_nass: str) -> pd.DataFrame:
    """Load county-level NASS yield records. Mirrors backend/models/train_yield.py."""
    parquet_dir = CACHE / 'county_parquets'
    t0 = time.time()
    if not list(parquet_dir.glob('*.parquet')):
        print(f'  Syncing NASS parquets from S3... (first run only, ~3 min)')
        _sync_prefix(CONFIG['data']['s3_bucket'], CONFIG['data']['nass_prefix'], parquet_dir)
    print(f'  NASS parquet dir: {parquet_dir} ({sum(1 for _ in parquet_dir.glob("*.parquet"))} files)')

    rows = []
    for pq in sorted(parquet_dir.glob('*.parquet')):
        if pq.name.startswith('_'):
            continue
        try:
            df = pd.read_parquet(pq)
        except Exception as exc:
            print(f'  WARN: skip {pq.name}: {exc}')
            continue

        # Wheat keeps WINTER/SPRING/ALL CLASSES; others use ALL CLASSES only.
        if crop_nass == 'WHEAT':
            class_ok = df['class_desc'].isin(['ALL CLASSES', 'WINTER', 'SPRING, (EXCL DURUM)'])
        else:
            class_ok = df['class_desc'] == 'ALL CLASSES'

        sub = df[
            (df['agg_level_desc'] == 'COUNTY') &
            (df['statisticcat_desc'] == 'YIELD') &
            (df['commodity_desc'] == crop_nass) &
            class_ok &
            (df['unit_desc'].str.contains('BU / ACRE', case=False, na=False)) &
            (df['domain_desc'] == 'TOTAL') &
            (df['freq_desc'] == 'ANNUAL')
        ].copy()
        if sub.empty:
            continue

        sub['fips'] = (
            sub['state_fips_code'].astype(str).str.zfill(2) +
            sub['county_code'].astype(str).str.zfill(3)
        )
        for _, r in sub.iterrows():
            try:
                val = float(str(r.get('value_num', r.get('Value', ''))).replace(',', ''))
            except (ValueError, TypeError):
                continue
            if val <= 0 or np.isnan(val):
                continue
            rows.append({
                'fips': r['fips'],
                'year': int(r['year']),
                'yield_bu': val,
                'state_fips': r['fips'][:2],
            })

    result = (
        pd.DataFrame(rows)
          .drop_duplicates(subset=['fips', 'year'], keep='last')
          .reset_index(drop=True)
    )
    print(f'  Loaded {len(result):,} NASS county-year rows for {crop_nass} ({time.time()-t0:.1f}s)')
    return result

nass_yields = load_nass_county_yields(CROP_NASS)
display(nass_yields.describe(include='all').T.head(10))

In [ ]:
# 3.3 — Load weather data (GHCN processed parquet + PRISM normals)
#
# If the S3 object isn't accessible with your credentials, set use_weather_features=False
# in CONFIG and skip this cell. The model will still train on NASS-only features.

def load_weather_from_s3() -> pd.DataFrame:
    local = CACHE / 'county_weather.parquet'
    if not local.exists():
        print('  Downloading GHCN processed parquet (~300 MB) ...')
        try:
            s3.download_file(CONFIG['data']['s3_bucket'], CONFIG['data']['weather_key'], str(local))
        except Exception as exc:
            print(f'  WARN: weather download failed: {exc}')
            return pd.DataFrame()
    df = pd.read_parquet(local)
    print(f'  Loaded GHCN: {len(df):,} rows')
    return df

def load_prism_from_s3() -> dict:
    local = CACHE / 'prism_normals.csv'
    if not local.exists():
        try:
            s3.download_file(CONFIG['data']['s3_bucket'], CONFIG['data']['prism_key'], str(local))
        except Exception as exc:
            print(f'  WARN: PRISM download failed: {exc}')
            return {}
    result = {}
    with open(local, 'r') as f:
        import csv
        reader = csv.DictReader(f)
        for row in reader:
            result[(row['fips'], int(row['month']))] = float(row['precip_normal_in'])
    print(f'  Loaded PRISM normals: {len(result):,} (fips,month) pairs')
    return result

if CONFIG['use_weather_features']:
    weather_df = load_weather_from_s3()
    prism_normals = load_prism_from_s3()
else:
    weather_df, prism_normals = pd.DataFrame(), {}

print(f'Weather enabled: {CONFIG["use_weather_features"]}, weather rows: {len(weather_df):,}, PRISM: {len(prism_normals):,}')

## 4 · Feature engineering

Two groups of features — matches `backend/models/train_yield.py` exactly so trained models are apples-to-apples with production:

**NASS historical (always):**
- `county_mean_yield` — mean of prior years for this county
- `county_yield_trend` — slope of prior years
- `prior_year_yield` — last year's realized yield
- `county_yield_std` — stdev of prior years
- `year` — trend capture

**Weather (if available):**
- `gdd_ytd` — accumulated growing degree days since planting
- `precip_season_in` — cumulative precipitation since planting
- `precip_deficit_in` — actual − normal precipitation
- `tmax_avg` — mean daily max temp
- `hot_days` — days with TMAX > 95°F

In [ ]:
# 4.1 — Weather feature computation (canonical implementation from train_yield.py)

def compute_weather_features(
    weather_df: pd.DataFrame,
    prism: dict,
    crop: str,
    fips_list: list,
    year_range: range,
    week: int,
) -> pd.DataFrame:
    """Return DataFrame with (fips, year) -> 5 weather features at the end of `week`."""
    if weather_df.empty:
        return pd.DataFrame()

    planting_doy = {'corn': 110, 'soybean': 125, 'wheat': 60}.get(crop, 110)
    wx = weather_df[weather_df['fips'].isin(fips_list)].copy()
    if wx.empty:
        return pd.DataFrame()

    wx['date_parsed'] = pd.to_datetime(wx['date'], format='%Y-%m-%d', errors='coerce')
    wx['year_val'] = wx['date_parsed'].dt.year
    wx['doy'] = wx['date_parsed'].dt.dayofyear

    rows = []
    for year in year_range:
        season_start = planting_doy
        season_end = planting_doy + week * 7
        wx_year = wx[wx['year_val'] == year]
        season = wx_year[(wx_year['doy'] >= season_start) & (wx_year['doy'] <= season_end)]
        if season.empty:
            continue
        for fips, grp in season.groupby('fips'):
            valid = grp.dropna(subset=['tmax_f', 'tmin_f'])
            prcp = grp['prcp_in'].dropna()
            if len(valid) < 5:
                continue
            tmax = valid['tmax_f'].values
            tmin = valid['tmin_f'].values
            gdd = ((tmax + tmin) / 2 - 50).clip(min=0).sum()
            precip_total = prcp.sum()

            # Compare to PRISM normal
            normal = 0.0
            start = date(year, 1, 1) + timedelta(days=season_start - 1)
            end = date(year, 1, 1) + timedelta(days=min(season_end, 365) - 1)
            cur = start
            while cur <= end:
                normal += prism.get((fips, cur.month), 0.25) / 30.0
                cur += timedelta(days=1)

            rows.append({
                'fips': fips, 'year': year,
                'gdd_ytd': round(float(gdd), 1),
                'precip_season_in': round(float(precip_total), 2),
                'precip_deficit_in': round(float(precip_total - normal), 2),
                'tmax_avg': round(float(tmax.mean()), 1),
                'hot_days': int((tmax > 95).sum()),
            })
    return pd.DataFrame(rows)

In [ ]:
# 4.2 — Full feature matrix builder

def _trend(series: pd.Series) -> float:
    if len(series) < 3:
        return 0.0
    try:
        return round(float(np.polyfit(np.arange(len(series)), series.values, 1)[0]), 2)
    except (np.linalg.LinAlgError, ValueError):
        return 0.0

def build_feature_matrix(
    nass_yields: pd.DataFrame,
    weather_df: pd.DataFrame,
    prism: dict,
    crop: str,
    week: int,
    use_weather: bool,
) -> pd.DataFrame:
    """Mirror of train_yield.py::train_single_model's feature-building stage."""
    fips_list = nass_yields['fips'].unique().tolist()
    year_range = range(
        max(int(nass_yields['year'].min()), 2000),
        int(nass_yields['year'].max()) + 1,
    )

    wx_lookup = None
    if use_weather and not weather_df.empty:
        wx = compute_weather_features(weather_df, prism, crop, fips_list, year_range, week)
        if not wx.empty:
            wx_lookup = wx.set_index(['fips', 'year'])
            print(f'  Built weather features: {len(wx):,} (fips,year) pairs at week {week}')

    rows = []
    for _, row in nass_yields.iterrows():
        fips, year, yld = row['fips'], row['year'], row['yield_bu']
        hist = nass_yields[(nass_yields['fips'] == fips) & (nass_yields['year'] < year)]['yield_bu']
        if len(hist) < 3:
            continue

        feat = {
            'county_mean_yield': hist.mean(),
            'county_yield_trend': _trend(hist),
            'prior_year_yield': hist.iloc[-1],
            'county_yield_std': hist.std(),
            'year': year,
            '_fips': fips,
            '_year': year,
            '_yield': yld,
        }
        if wx_lookup is not None and (fips, year) in wx_lookup.index:
            wxr = wx_lookup.loc[(fips, year)]
            feat.update({
                'gdd_ytd': wxr['gdd_ytd'],
                'precip_season_in': wxr['precip_season_in'],
                'precip_deficit_in': wxr['precip_deficit_in'],
                'tmax_avg': wxr['tmax_avg'],
                'hot_days': wxr['hot_days'],
            })
        rows.append(feat)

    df = pd.DataFrame(rows)
    print(f'  Feature matrix: {len(df):,} rows × {len([c for c in df.columns if not c.startswith("_")])} features')
    return df

def split_walk_forward(df: pd.DataFrame, train_end: int, val_years: tuple, test_years: tuple):
    """Split into train/val/test by year. Returns 6 objects: X_tr, y_tr, X_val, y_val, X_te, y_te."""
    feat_cols = [c for c in df.columns if not c.startswith('_') and df[c].notna().any()]
    tr = df[df['_year'] <= train_end]
    va = df[(df['_year'] >= val_years[0]) & (df['_year'] <= val_years[1])]
    te = df[(df['_year'] >= test_years[0]) & (df['_year'] <= test_years[1])]
    return (
        tr[feat_cols], tr['_yield'],
        va[feat_cols], va['_yield'],
        te[feat_cols], te['_yield'],
        feat_cols,
    )

## 5 · Baselines

A yield model is only interesting if it beats the dumb model. We compute two baselines on the val years — any submission should at minimum clear `county_mean_rrmse * 0.9` on the test set (the 10% gate).

Production RRMSE reference (corn, 2023–2024 test):
- County historical mean: **23.78%**
- Prior year: **16.06%**
- Current production (LightGBM quantile, 10 features): **17.44–18.51%** depending on week

In [ ]:
# Baselines are computed once per (crop, val-range) — independent of the week we're training.

baselines = compute_baselines(nass_yields, list(range(CONFIG['val_years'][0], CONFIG['val_years'][1] + 1)))
print('Baselines on val years:', baselines)

GATE_RRMSE = baselines.get('county_mean_rrmse', 100) * (1 - CONFIG['baseline_gate_pct'] / 100)
print(f'Gate (county_mean × {1 - CONFIG["baseline_gate_pct"]/100:.2f}): {GATE_RRMSE:.2f}%')

## 6 · Model factory

All models share a uniform interface:

```python
build_model(hyperparams) -> {'q10': estimator, 'q50': estimator, 'q90': estimator}
```

…where each estimator is a scikit-learn-compatible regressor with `.fit()`, `.predict()`, `.feature_importances_` (optional). This matches the tri-quantile structure of the production `YieldModel`.

**To add your own model family**, add an entry to `MODEL_FACTORY`. A worked example of a stacked ensemble is in Section 6.3.

In [ ]:
# 6.1 — Built-in model families

def build_lightgbm_quantile(hp: dict) -> dict:
    """LightGBM with three separate quantile regressors (production default)."""
    from lightgbm import LGBMRegressor
    shared = {**hp, 'verbose': -1, 'n_jobs': -1}
    return {
        'q10': LGBMRegressor(objective='quantile', alpha=0.10, **shared),
        'q50': LGBMRegressor(objective='quantile', alpha=0.50, **shared),
        'q90': LGBMRegressor(objective='quantile', alpha=0.90, **shared),
    }

def build_xgboost_quantile(hp: dict) -> dict:
    """XGBoost quantile regression (requires xgboost >= 2.0)."""
    from xgboost import XGBRegressor
    shared = {k: v for k, v in hp.items() if k not in ('min_child_samples',)}
    shared.setdefault('tree_method', 'hist')
    return {
        'q10': XGBRegressor(objective='reg:quantileerror', quantile_alpha=0.10, **shared),
        'q50': XGBRegressor(objective='reg:quantileerror', quantile_alpha=0.50, **shared),
        'q90': XGBRegressor(objective='reg:quantileerror', quantile_alpha=0.90, **shared),
    }

def build_catboost_quantile(hp: dict) -> dict:
    """CatBoost quantile regression — often wins on tabular with heavy categorical handling."""
    from catboost import CatBoostRegressor
    shared = {
        'iterations': hp.get('n_estimators', 500),
        'depth': hp.get('max_depth', 5),
        'learning_rate': hp.get('learning_rate', 0.04),
        'verbose': 0,
    }
    return {
        'q10': CatBoostRegressor(loss_function='Quantile:alpha=0.10', **shared),
        'q50': CatBoostRegressor(loss_function='Quantile:alpha=0.50', **shared),
        'q90': CatBoostRegressor(loss_function='Quantile:alpha=0.90', **shared),
    }

def build_random_forest_quantile(hp: dict) -> dict:
    """Single RandomForest — emit p50 directly, get p10/p90 via tree-level quantiles.

    Cheap baseline. Only p50 is 'trained' — the 10/90 quantiles come from the ensemble's
    per-tree predictions at inference time. Wrapped for API compat below.
    """
    from sklearn.ensemble import RandomForestRegressor
    shared = {
        'n_estimators': hp.get('n_estimators', 500),
        'max_depth': hp.get('max_depth', None),
        'min_samples_leaf': hp.get('min_child_samples', 20),
        'n_jobs': -1,
    }
    rf = RandomForestRegressor(**shared)

    class _TreeQuantileView:
        def __init__(self, rf, q):
            self._rf, self._q = rf, q
            self.feature_importances_ = None  # set after fit
        def fit(self, X, y):
            if not hasattr(self._rf, 'estimators_'):
                self._rf.fit(X, y)
            self.feature_importances_ = self._rf.feature_importances_
            return self
        def predict(self, X):
            per_tree = np.stack([t.predict(X) for t in self._rf.estimators_], axis=1)
            return np.quantile(per_tree, self._q, axis=1)

    return {'q10': _TreeQuantileView(rf, 0.10), 'q50': _TreeQuantileView(rf, 0.50), 'q90': _TreeQuantileView(rf, 0.90)}

def build_histgb_quantile(hp: dict) -> dict:
    """sklearn HistGradientBoostingRegressor quantile (no external deps beyond sklearn)."""
    from sklearn.ensemble import HistGradientBoostingRegressor
    shared = {
        'max_iter': hp.get('n_estimators', 500),
        'max_depth': hp.get('max_depth', 5),
        'learning_rate': hp.get('learning_rate', 0.04),
        'min_samples_leaf': hp.get('min_child_samples', 20),
    }
    return {
        'q10': HistGradientBoostingRegressor(loss='quantile', quantile=0.10, **shared),
        'q50': HistGradientBoostingRegressor(loss='quantile', quantile=0.50, **shared),
        'q90': HistGradientBoostingRegressor(loss='quantile', quantile=0.90, **shared),
    }

MODEL_FACTORY: dict[str, Callable[[dict], dict]] = {
    'lightgbm_quantile': build_lightgbm_quantile,
    'xgboost_quantile': build_xgboost_quantile,
    'catboost_quantile': build_catboost_quantile,
    'random_forest_quantile': build_random_forest_quantile,
    'histgb_quantile': build_histgb_quantile,
}

print('Registered model families:', list(MODEL_FACTORY.keys()))

In [ ]:
# 6.2 — Wrap any factory output as a backend-compatible YieldModel.
#
# The YieldModel dataclass expects q10/q50/q90 attributes with .predict() and .feature_importances_.
# All of our factories emit dicts with exactly that shape — so wrapping is a matter of slotting
# them in to an empty YieldModel and running its .fit() pipeline.

def train_yield_model(
    model_family: str,
    hyperparams: dict,
    crop: str,
    week: int,
    X_tr: pd.DataFrame, y_tr: pd.Series,
    X_va: pd.DataFrame, y_va: pd.Series,
    feat_cols: list,
) -> tuple[YieldModel, dict]:
    """Train one YieldModel for (crop, week). Returns (model, metrics_dict)."""
    builder = MODEL_FACTORY.get(model_family)
    if builder is None:
        raise KeyError(f'Unknown model_family {model_family!r}. Known: {list(MODEL_FACTORY.keys())}')

    estimators = builder(hyperparams)

    # Build a YieldModel shell and stuff our estimators into it
    m = YieldModel(
        crop=crop,
        week=week,
        model_ver=date.today().isoformat(),
        feature_cols=list(feat_cols),
    )

    # Clean NaNs the same way YieldModel.fit() does
    Xtr_c = X_tr[feat_cols].fillna(X_tr[feat_cols].median())
    Xva_c = X_va[feat_cols].fillna(Xtr_c.median())  # use train median for val

    m.train_median_yield = float(y_tr.median())
    m.q10 = estimators['q10'].fit(Xtr_c, y_tr)
    m.q50 = estimators['q50'].fit(Xtr_c, y_tr)
    m.q90 = estimators['q90'].fit(Xtr_c, y_tr)

    # Conformal calibration on val
    m.calibrate_conformal(X_va, y_va)

    # Evaluate
    train_preds = np.asarray(m.q50.predict(Xtr_c))
    val_preds = np.asarray(m.q50.predict(Xva_c))
    train_rrmse = compute_rrmse(y_tr.values, train_preds)
    val_rrmse = compute_rrmse(y_va.values, val_preds)

    metrics = {
        'train_rrmse': train_rrmse,
        'val_rrmse': val_rrmse,
        'n_train': len(X_tr),
        'n_val': len(X_va),
        'feature_cols': list(feat_cols),
        'n_features': len(feat_cols),
        'conformal_q80': m.conformal_q80,
    }
    return m, metrics

In [ ]:
# 6.3 — Worked example: a stacked ensemble. Copy + modify this cell to register your own.

def build_stacked_lgbm_xgb(hp: dict) -> dict:
    """Average LightGBM and XGBoost predictions at each quantile. Trivial stacking baseline."""
    lgb = build_lightgbm_quantile(hp)
    xgb = build_xgboost_quantile(hp)

    class _AvgOfTwo:
        def __init__(self, a, b):
            self._a, self._b = a, b
            self.feature_importances_ = None
        def fit(self, X, y):
            self._a.fit(X, y)
            self._b.fit(X, y)
            # Average importances for visualization; predictions are averaged at inference time
            if hasattr(self._a, 'feature_importances_') and hasattr(self._b, 'feature_importances_'):
                self.feature_importances_ = (np.asarray(self._a.feature_importances_) +
                                              np.asarray(self._b.feature_importances_)) / 2
            return self
        def predict(self, X):
            return (np.asarray(self._a.predict(X)) + np.asarray(self._b.predict(X))) / 2

    return {
        'q10': _AvgOfTwo(lgb['q10'], xgb['q10']),
        'q50': _AvgOfTwo(lgb['q50'], xgb['q50']),
        'q90': _AvgOfTwo(lgb['q90'], xgb['q90']),
    }

MODEL_FACTORY['stacked_lgbm_xgb'] = build_stacked_lgbm_xgb
print('Now registered:', list(MODEL_FACTORY.keys()))

## 7 · Train + log to MLflow

One MLflow run per (crop, week). Parent run aggregates. Artifacts logged:
- Params (hyperparams, feature set, split years)
- Metrics (train/val/test RRMSE, baseline RRMSEs, gap vs gate, coverage@80)
- Model pickle (as an MLflow artifact, *plus* a separate `model.pkl` in the notebook's local artifact dir for backend drop-in)
- `metrics.json` with the schema the backend expects

In [ ]:
# 7.1 — The training loop

ARTIFACT_ROOT = Path('./artifact_export')
ARTIFACT_ROOT.mkdir(exist_ok=True)

results = []  # one dict per (crop, week)

parent_run = mlflow.start_run(run_name=f'{CROP}_{CONFIG["model_family"]}_{date.today().isoformat()}')
mlflow.log_params({
    'crop': CROP,
    'model_family': CONFIG['model_family'],
    'use_weather_features': CONFIG['use_weather_features'],
    'train_end_year': CONFIG['train_end_year'],
    'val_years': f"{CONFIG['val_years'][0]}-{CONFIG['val_years'][1]}",
    'test_years': f"{CONFIG['test_years'][0]}-{CONFIG['test_years'][1]}",
    'experimenter': CONFIG['experimenter'],
    **{f'hp.{k}': v for k, v in CONFIG['hyperparams'].items()},
})

try:
    for week in CONFIG['weeks']:
        run_name = f'{CROP}_week{week}_{CONFIG["model_family"]}'
        with mlflow.start_run(run_name=run_name, nested=True) as run:
            mlflow.log_param('week', week)

            # Build feature matrix for this week
            t0 = time.time()
            df = build_feature_matrix(
                nass_yields, weather_df, prism_normals,
                CROP, week, CONFIG['use_weather_features'],
            )
            X_tr, y_tr, X_va, y_va, X_te, y_te, feat_cols = split_walk_forward(
                df, CONFIG['train_end_year'], CONFIG['val_years'], CONFIG['test_years'],
            )
            print(f'[week {week}] train={len(X_tr)} val={len(X_va)} test={len(X_te)} feats={len(feat_cols)}')

            if len(X_tr) < 50 or len(X_va) < 10:
                print(f'  SKIP week {week}: insufficient data')
                continue

            # Train
            m, train_metrics = train_yield_model(
                CONFIG['model_family'], CONFIG['hyperparams'],
                CROP, week, X_tr, y_tr, X_va, y_va, feat_cols,
            )

            # Test-set evaluation
            test_rrmse = None
            test_coverage = None
            if len(X_te) > 0:
                Xte_c = X_te[feat_cols].fillna(X_tr[feat_cols].median())
                p50 = np.asarray(m.q50.predict(Xte_c))
                p10 = np.asarray(m.q10.predict(Xte_c))
                p90 = np.asarray(m.q90.predict(Xte_c))
                # Enforce ordering
                stacked = np.stack([p10, p50, p90], axis=1)
                stacked.sort(axis=1)
                p10, p50, p90 = stacked[:, 0], stacked[:, 1], stacked[:, 2]
                test_rrmse = compute_rrmse(y_te.values, p50)
                test_coverage = float(((y_te.values >= p10) & (y_te.values <= p90)).mean())

            # Baseline gate
            county_mean_baseline = baselines.get('county_mean_rrmse', 100)
            beats_gate = (
                test_rrmse is not None and
                test_rrmse < county_mean_baseline * (1 - CONFIG['baseline_gate_pct'] / 100)
            )

            # Top features (by p50 importance)
            top = m.get_top_features(7) if hasattr(m, 'get_top_features') else []

            # Collect metrics in the canonical schema
            full_metrics = {
                'crop': CROP,
                'week': week,
                'model_ver': m.model_ver,
                'train_rrmse': train_metrics['train_rrmse'],
                'val_rrmse': train_metrics['val_rrmse'],
                'test_rrmse': test_rrmse,
                'baselines': baselines,
                'beats_baseline': beats_gate,
                'n_train': train_metrics['n_train'],
                'n_val': train_metrics['n_val'],
                'n_test': len(X_te),
                'n_features': len(feat_cols),
                'feature_cols': feat_cols,
                'has_weather_features': CONFIG['use_weather_features'] and any('gdd_ytd' == c for c in feat_cols),
                'top_features': [{'name': n, 'importance': round(float(i), 4)} for n, i in top],
                'conformal_q80': train_metrics['conformal_q80'],
                'confidence_tier': YieldModel.confidence_tier(week),
                'test_coverage_80': test_coverage,
                'experiment_meta': {
                    'notebook_user': CONFIG['experimenter'],
                    'model_family': CONFIG['model_family'],
                    'hyperparams': CONFIG['hyperparams'],
                    'mlflow_run_id': run.info.run_id,
                    'train_time_sec': round(time.time() - t0, 1),
                },
            }

            # Log to MLflow
            mlflow.log_metrics({
                'train_rrmse': full_metrics['train_rrmse'],
                'val_rrmse': full_metrics['val_rrmse'],
                'test_rrmse': test_rrmse or -1,
                'test_coverage_80': test_coverage or -1,
                'county_mean_baseline_rrmse': county_mean_baseline,
                'prior_year_baseline_rrmse': baselines.get('prior_year_rrmse', -1),
                'gap_vs_gate': (test_rrmse or 999) - county_mean_baseline * (1 - CONFIG['baseline_gate_pct'] / 100),
                'beats_gate': 1.0 if beats_gate else 0.0,
            })

            # Pickle the YieldModel and save locally + log as MLflow artifact.
            # Local path matches production: backend/artifacts/yield/{crop}/week_{week}/model.pkl
            local_dir = ARTIFACT_ROOT / CROP / f'week_{week}'
            local_dir.mkdir(parents=True, exist_ok=True)
            pkl_path = local_dir / 'model.pkl'
            metrics_path = local_dir / 'metrics.json'

            # Use YieldModel.save() if MODEL_SIGNING_KEY is set, else plain pickle (no sig)
            try:
                m.save(pkl_path)
            except Exception:
                # Some signing error — write unsigned. Maintainer will sign during adoption.
                with open(pkl_path, 'wb') as f:
                    pickle.dump(m, f, protocol=pickle.HIGHEST_PROTOCOL)

            with open(metrics_path, 'w') as f:
                json.dump(full_metrics, f, indent=2, default=str)

            mlflow.log_artifact(str(pkl_path), artifact_path=f'{CROP}/week_{week}')
            mlflow.log_artifact(str(metrics_path), artifact_path=f'{CROP}/week_{week}')

            # Console summary
            gate_flag = '✅ BEATS GATE' if beats_gate else '❌ below gate'
            print(
                f'  [week {week}] train={full_metrics["train_rrmse"]:.2f}%  '
                f'val={full_metrics["val_rrmse"]:.2f}%  '
                f'test={test_rrmse:.2f}% {gate_flag}  '
                f'gate={GATE_RRMSE:.2f}%  cov@80={test_coverage:.2f}' if test_rrmse else f'  [week {week}] done (no test set)'
            )
            results.append(full_metrics)
finally:
    mlflow.end_run()

print(f'\n=== Done. Trained {len(results)} (crop, week) combos. MLflow runs under experiment {CONFIG["mlflow_experiment"]!r}. ===')

## 8 · Results summary

Cross-run comparison. If MLflow is local, the dataframe below *is* the comparison table; if MLflow is remote, the UI at your tracking URI will have richer sorting/filtering.

In [ ]:
summary = pd.DataFrame([
    {
        'crop': r['crop'],
        'week': r['week'],
        'model_family': r['experiment_meta']['model_family'],
        'train_rrmse': r['train_rrmse'],
        'val_rrmse': r['val_rrmse'],
        'test_rrmse': r['test_rrmse'],
        'county_mean_baseline': r['baselines'].get('county_mean_rrmse'),
        'prior_year_baseline': r['baselines'].get('prior_year_rrmse'),
        'beats_gate': r['beats_baseline'],
        'coverage@80': r.get('test_coverage_80'),
        'n_features': r['n_features'],
        'run_time_s': r['experiment_meta']['train_time_sec'],
    }
    for r in results
])
display(summary)

In [ ]:
# 8.1 — Compare against ALL past runs (across notebook sessions) via MLflow

runs = mlflow.search_runs(
    experiment_names=[CONFIG['mlflow_experiment']],
    max_results=100,
    order_by=['metrics.test_rrmse ASC'],
)
# Show the leaderboard: lower test_rrmse is better
leaderboard_cols = [c for c in [
    'tags.mlflow.runName', 'params.week', 'params.model_family',
    'metrics.test_rrmse', 'metrics.val_rrmse', 'metrics.test_coverage_80',
    'metrics.beats_gate', 'params.hp.n_estimators', 'params.hp.learning_rate',
    'start_time',
] if c in runs.columns]
display(runs[leaderboard_cols].head(20) if not runs.empty else 'No prior runs yet.')

## 9 · Export backend-compatible artifact

Already done in Section 7 as a side effect (every trained model is saved to `./artifact_export/{crop}/week_{N}/`). This cell is a **sanity check** — it reloads the pickle through the production `YieldModel.load()` path to confirm the artifact is adoptable.

In [ ]:
# Verify: reload each artifact through YieldModel.load(), predict on a sample row, confirm the shape.

verified = []
for r in results:
    pkl_path = ARTIFACT_ROOT / r['crop'] / f'week_{r["week"]}' / 'model.pkl'
    try:
        m_reloaded = YieldModel.load(pkl_path)
    except Exception as exc:
        print(f'  ❌ {pkl_path}: load failed — {exc}')
        continue

    # Build one sample row matching the feature schema and predict
    sample = pd.DataFrame([{c: 0.0 for c in r['feature_cols']}])
    try:
        pred = m_reloaded.predict(sample)
        ok = (
            'p10' in pred and 'p50' in pred and 'p90' in pred and
            pred['p10'] <= pred['p50'] <= pred['p90']
        )
    except Exception as exc:
        print(f'  ❌ {pkl_path}: predict failed — {exc}')
        continue

    verified.append({
        'crop': r['crop'], 'week': r['week'],
        'pkl_size_kb': pkl_path.stat().st_size // 1024,
        'predict_ok': ok,
        'beats_gate': r['beats_baseline'],
        'test_rrmse': r['test_rrmse'],
    })

verif_df = pd.DataFrame(verified)
display(verif_df)

if not verif_df.empty:
    adoptable = verif_df[(verif_df['predict_ok']) & (verif_df['beats_gate'])]
    print(f'\n{len(adoptable)}/{len(verif_df)} models are adoptable (predict OK + beats gate).')
    if not adoptable.empty:
        print('\nTo submit to maintainers, zip the artifact_export/ directory:')
        print(f'    cd {ARTIFACT_ROOT.parent} && zip -r yield_submission_{CROP}_{date.today().isoformat()}.zip artifact_export/')
        print('\nOr upload directly to S3 (requires maintainer credentials):')
        print(f'    aws s3 sync artifact_export/ s3://usda-analysis-datasets/models/yield/ --exclude "*" --include "*.pkl" --include "metrics.json"')

## 10 · (Optional) Upload to S3

Only run this cell if you are a maintainer with write credentials to `s3://usda-analysis-datasets/models/yield/`. Most collaborators should stop at Section 9 and submit the zipped `artifact_export/` directory via PR.

In [ ]:
MAINTAINER_UPLOAD = False  # set True to actually upload

if MAINTAINER_UPLOAD:
    # Maintainers: ensure MODEL_SIGNING_KEY is set so .save() writes the .sig sidecar.
    if not os.environ.get('MODEL_SIGNING_KEY'):
        print('❌ Refusing to upload: MODEL_SIGNING_KEY not set. Signed artifacts required for production.')
    else:
        # Re-save every model so .sig sidecars are generated with the current key
        for r in results:
            pkl_path = ARTIFACT_ROOT / r['crop'] / f'week_{r["week"]}' / 'model.pkl'
            m_reloaded = YieldModel.load(pkl_path)
            m_reloaded.save(pkl_path)

        # Sync to S3
        import subprocess
        subprocess.run([
            'aws', 's3', 'sync',
            str(ARTIFACT_ROOT) + '/',
            f's3://{CONFIG["data"]["s3_bucket"]}/models/yield/',
        ], check=True)
        print(f'✅ Uploaded to s3://{CONFIG["data"]["s3_bucket"]}/models/yield/')
        print('  Run `sudo systemctl restart ag-prediction` on EC2 to pick up the new artifacts.')
else:
    print('MAINTAINER_UPLOAD is False — skipping. Set to True above if you have write credentials.')

## Appendix A · Debugging a bad result

If your model fails the gate (test_rrmse ≥ 21.4% for corn), work through this checklist in order. Each item has at least once been the actual culprit:

1. **Leakage.** Is any feature computed using data from the test year? The canonical features all use `year < current_year` — a custom feature that forgot to subset is the single most common bug.
2. **Underfit.** Train RRMSE close to val RRMSE and both high? Increase `n_estimators` or `max_depth`, or reduce `min_child_samples`.
3. **Overfit.** Train RRMSE much lower than val RRMSE? Lower `learning_rate` + raise `min_child_samples`, or add `reg_alpha` / `reg_lambda`.
4. **Weather features absent.** Check that Section 3.3 ran cleanly — if `weather_df` is empty you're training on 5 NASS features only, which maxes out around val 18–19% on corn.
5. **Wrong planting DOY for the region.** The built-in dates are Corn Belt averages. Southern corn and northern corn have 30+ day differences. Consider a per-state planting DOY if you're focused on a specific region.
6. **Test years are anomalous.** 2023–2024 had atypical weather patterns in some regions. This is a real-world problem, not a model problem — consider reporting which test years drove the failure.

## Appendix B · Adding a new feature

Say you want to try adding soil moisture from SMAP. The cleanest path:

1. Extend `compute_weather_features()` (or add a parallel `compute_soil_moisture_features()`) to return the new column keyed by (fips, year).
2. Merge it into the feature DataFrame in `build_feature_matrix()` just like the weather features.
3. Add the feature name to `backend/models/yield_model.py::FEATURE_LABELS` (via PR) so the backend can name-drop it in the "key driver" UI. **Until that lands, the backend will still accept the model** — unknown feature names are silently treated as opaque.
4. Train and compare. The MLflow leaderboard in Section 8.1 will let you compare the new-feature model head-to-head against the current baseline.

## Appendix C · Reproducing production numbers

To exactly reproduce the production corn model on week 10:

```python
CONFIG['model_family'] = 'lightgbm_quantile'
CONFIG['weeks'] = [10]
CONFIG['use_weather_features'] = True
CONFIG['hyperparams'] = {
    'n_estimators': 500, 'max_depth': 5, 'learning_rate': 0.04,
    'min_child_samples': 20, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_alpha': 0.1,
}
```

Expected result: `test_rrmse ≈ 17.8%`, beats the 21.4% gate comfortably.